# PNEUMYST AI · Exploratory Data Analysis

Run with the `.venv` kernel. Charts are also saved to `docs/figures/eda/`.

Section 10 shows real X-rays: clear outputs before committing.
`jupyter nbconvert --clear-output --inplace notebooks/eda.ipynb`

In [ ]:
import json, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from dotenv import load_dotenv
load_dotenv(REPO / ".env")

%matplotlib inline
import pandas as pd
from IPython.display import display

import ml.eda.run_eda as eda
from ml.config import data_root

eda.SHOW = True
ROOT = data_root()
OUT = REPO / "docs" / "figures" / "eda"
LOCAL = ROOT / "processed" / "eda"
OUT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

dfs = eda.load_manifests(ROOT / "processed" / "manifests")
nih = dfs["nih"]
nih_train = nih[nih["split"] == "train"]
{name: len(df) for name, df in dfs.items()}

## 1. Images per dataset and split

In [ ]:
pd.DataFrame(eda.plot_split_sizes(dfs, OUT)).fillna(0).astype(int)

## 2. Tuberculosis label balance

In [ ]:
pd.DataFrame(eda.plot_tb_balance(dfs, OUT)).T

## 3. NIH prevalence by split

In [ ]:
prev = eda.plot_nih_prevalence(nih, OUT)
(pd.DataFrame(prev["percent"])
   .assign(test_to_train=pd.Series(prev["test_to_train_ratio"]))
   .sort_values("test_to_train", ascending=False))

## 4. Which findings occur together

In [ ]:
pd.DataFrame(eda.plot_nih_cooccurrence(nih_train, OUT))

## 5. Findings per image

In [ ]:
pd.Series(eda.plot_labels_per_image(nih_train, OUT), name="images").to_frame()

## 6. AP-view shortcut check

In [ ]:
view = eda.plot_view_shortcut(nih_train, OUT)
(pd.Series(view["ap_percent_among_positives"], name="% AP")
   .sort_values(ascending=False).to_frame()
   .assign(vs_overall=lambda d: (d["% AP"] - view["overall_ap_percent"]).round(1)))

## 7. Demographics

In [ ]:
pd.DataFrame(eda.plot_demographics(dfs, OUT)).T

## 8. NIH images per patient

In [ ]:
print(eda.plot_followups(nih, OUT))
(nih.groupby("split")
    .agg(images=("image_id", "size"), patients=("patient_id", "nunique"))
    .assign(images_per_patient=lambda d: (d.images / d.patients).round(1)))

## 9. Image properties (sampled)

Raise `N` for a bigger sample (slower).

In [ ]:
N = 300
stats = eda.sample_image_stats(dfs, ROOT, N, seed=42)
pd.DataFrame(eda.plot_image_stats(stats, OUT)).T

## 10. Sample X-rays (local only)

Check for borders, inverted images and burned-in text.

In [ ]:
eda.sample_grids(dfs, ROOT, LOCAL, seed=42)

---
# Part 2 · Checks that set preprocessing

In [ ]:
import ml.eda.checks as checks

SAMPLE = 200  # images per source/mode for sampled checks
extra = {}

## 11. Image modes and real grayscale

Census of every image's mode (header only, fast), then: are RGB / palette / RGBA images really gray?

In [ ]:
census = checks.mode_census(dfs, ROOT)
display(census.groupby(["source", "mode"]).size().unstack(fill_value=0))

channels = checks.channel_check(census, ROOT, OUT, n=SAMPLE)
extra["color_modes"] = channels.groupby(["source", "mode"]).agg(
    sampled=("is_gray", "size"), gray_percent=("is_gray", lambda s: round(s.mean() * 100, 1)),
    max_channel_diff=("max_channel_diff", "max"), min_alpha=("alpha_min", "min")).reset_index()
extra["color_modes"]

## 12. Inverted images

Palette check (P images) plus a brightness heuristic: inverted X-rays have bright corners.

In [ ]:
inv = checks.inversion_check(census, ROOT, OUT, LOCAL, n=SAMPLE)
extra["inversion"] = inv.groupby("source").agg(
    sampled=("suspect_inverted", "size"), suspect_inverted=("suspect_inverted", "sum"),
    palette_inverted=("palette_inverted", lambda s: int(s.fillna(False).sum()))).reset_index()
extra["inversion"]

## 13. Borders and aspect ratio

In [ ]:
borders = checks.border_check(dfs, ROOT, OUT, n=SAMPLE)
extra["borders"] = borders.groupby("source")[["border_frac", "black_frac", "white_frac", "aspect"]].median().round(3)
extra["borders"]

## 14. Lesion size vs training resolution

NIH boxes rescaled to the model input. Tiny boxes at 224 px argue for 320 px.

In [ ]:
extra["bbox_sizes"] = checks.bbox_sizes(ROOT, OUT)
extra["bbox_sizes"]

## 15. Duplicate pairs for review (local only)

Near-identical pairs that were removed, plus cross-split pairs that were kept.

In [ ]:
checks.duplicate_review(dfs, ROOT, LOCAL)

---
# Part 3 · Checks for the report

## 16. Prevalence by age, sex and view (NIH train)

In [ ]:
groups = checks.prevalence_by_group(nih_train, OUT)
display(groups["age_band"])
display(groups["sex"])
groups["view"]

## 17. Positivity vs follow-up number

In [ ]:
extra["followup"] = checks.followup_effect(nih, OUT)
extra["followup"]

## 18. Sharpness and exposure

The 1st / 99th percentiles become starting thresholds for the app's image-quality check.

In [ ]:
quality, thresholds = checks.quality_stats(dfs, ROOT, OUT, n=SAMPLE)
display(quality.groupby("source")[["sharpness", "mean", "dark_frac", "bright_frac"]].median().round(2))
extra["quality_thresholds"] = thresholds
thresholds

## 19. External readings that disagree with labels

In [ ]:
extra["reading_mismatches"] = checks.reading_mismatches(dfs, ROOT, LOCAL)
extra["reading_mismatches"]

## Save Part 2–3 summary

In [ ]:
def _jsonable(v):
    return v.reset_index().to_dict("records") if isinstance(v, pd.DataFrame) else v

(OUT / "eda_checks_summary.json").write_text(json.dumps({k: _jsonable(v) for k, v in extra.items()}, indent=2, default=str))
print("saved", OUT / "eda_checks_summary.json")